In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt

from statsmodels.stats.proportion import proportions_ztest

print("Libraries loaded.")

Libraries loaded.


##  Load the cleaned dataset

The analysis uses the cleaned dataset produced during the previous preprocessing task.

The original Hotel Booking Demand dataset required data cleaning before statistical analysis. Therefore, this notebook does not repeat the complete preprocessing procedure. Instead, it uses the prepared `hotel_bookings_clean.csv` file.

After loading the file, the dataset dimensions and the first observations are displayed to verify that the correct data have been loaded.

In [ ]:
from google.colab import files

# Upload the cleaned dataset
uploaded = files.upload()

# Get the uploaded file name
filename = next(iter(uploaded))

# Load the dataset
df = pd.read_csv(filename)

print(f"Loaded file: {filename}")
print(f"Dataset shape: {df.shape}")

display(df.head())

Saving hotel_bookings_clean(1).csv to hotel_bookings_clean(1) (1).csv
Loaded file: hotel_bookings_clean(1) (1).csv
Dataset shape: (119205, 34)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,total_nights,total_guests,previous_bookings,room_changed,season
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,0,Transient,0.0,0,0,0,2,0,0,Summer
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,0,Transient,0.0,0,0,0,2,0,0,Summer
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,0,Transient,75.0,0,0,1,1,0,1,Summer
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,0,Transient,75.0,0,0,1,1,0,0,Summer
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,0,Transient,98.0,0,1,2,2,0,0,Summer


##  Verification of variables used in hypothesis testing

Three variables are required for the planned statistical tests:

- `is_canceled` — indicates whether a booking was cancelled (`1`) or not cancelled (`0`),
- `lead_time` — number of days between the booking date and the arrival date,
- `hotel` — type of hotel (`City Hotel` or `Resort Hotel`).

Before performing the tests, the notebook verifies:

- whether all required variables exist,
- whether they contain missing values,
- whether the expected categories are present.

This step helps ensure that the statistical tests are performed on correctly prepared data.

In [ ]:
required_columns = [
    "is_canceled",
    "lead_time",
    "hotel"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("All required columns are available.")

print("\nMissing values:")
display(
    df[required_columns]
    .isna()
    .sum()
)

print(
    "\nCancellation status values:",
    sorted(df["is_canceled"].dropna().unique())
)

print(
    "Hotel types:",
    df["hotel"].dropna().unique()
)

All required columns are available.

Missing values:


,0
is_canceled,0
lead_time,0
hotel,0



Cancellation status values: [np.int64(0), np.int64(1)]
Hotel types: ['Resort Hotel' 'City Hotel']


## Significance level and reporting functions

For all main hypothesis tests, the significance level is set to:

$$
\alpha = 0.05
$$

The statistical decision is based on comparing the p-value with the significance level $\alpha$:

- if $p < 0.05$, the null hypothesis $H_0$ is **rejected**,
- if $p \geq 0.05$, the null hypothesis $H_0$ is **not rejected**.

The helper functions below standardize the presentation of the test statistic, p-value, significance level, and statistical decision for all tests.

This follows the same reporting approach as the example provided during classes.

In [ ]:
ALPHA = 0.05


def decision(p_value, alpha=ALPHA):
    return (
        "Reject H0"
        if p_value < alpha
        else "Do not reject H0"
    )


def format_p_value(p_value):
    if p_value == 0:
        return "< 1e-300"
    elif p_value < 0.0001:
        return f"{p_value:.3e}"
    else:
        return f"{p_value:.6f}"


def print_result(
    test_name,
    h0,
    h1,
    statistic_name,
    statistic,
    p_value,
    alpha=ALPHA
):
    print("=" * 80)
    print(test_name)
    print("-" * 80)

    print("H0:", h0)
    print("H1:", h1)

    print(f"\n{statistic_name} = {statistic:.4f}")
    print(f"p-value = {format_p_value(p_value)}")
    print(f"alpha = {alpha}")

    print("\nDecision:", decision(p_value, alpha))

# Test 1 — One-sample population proportion test

## Research question

Is the population proportion of cancelled hotel bookings significantly different from **35%**?

The variable used in this analysis is:

- `is_canceled`

where:

- `0` means that the booking was not cancelled,
- `1` means that the booking was cancelled.

A reference cancellation rate of **35%** is adopted for the purpose of the one-sample hypothesis test.

Let $p$ denote the population proportion of cancelled bookings.

The hypotheses are:

$$
H_0: p = 0.35
$$

$$
H_1: p \neq 0.35
$$

The alternative hypothesis is two-sided because the objective is to determine whether the cancellation rate is different from 35%, regardless of whether it is higher or lower.

## Why this test?

The variable of interest is binary and the hypothesis concerns **one population proportion compared with a predefined reference proportion**. Therefore, a **one-sample population proportion z-test** is appropriate.

Before performing the test, the observed number and percentage of cancelled and non-cancelled bookings are displayed.

## Descriptive statistics

The number and percentage of cancelled and non-cancelled bookings are calculated before the hypothesis test.

This provides context for interpreting the observed cancellation proportion.

### Descriptive statistics

The number and percentage of cancelled and non-cancelled bookings are calculated before the hypothesis test.

This provides context for interpreting the observed cancellation proportion.

In [ ]:
cancellation_counts = (
    df["is_canceled"]
    .value_counts()
    .sort_index()
)

cancellation_percentages = (
    df["is_canceled"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

print("Cancellation counts:")
display(cancellation_counts)

print("\nCancellation percentages:")
display(cancellation_percentages.round(2))

Cancellation counts:


,count
is_canceled,
0,75010
1,44195



Cancellation percentages:


,proportion
is_canceled,
0,62.93
1,37.07


### One-sample population proportion z-test

The observed cancellation proportion is compared with the reference value of 0.35.

The test returns:

- the z-statistic,
- the p-value.

The p-value is then compared with the significance level alpha = 0.05.

In [ ]:
# Reference cancellation rate
p0 = 0.35

# Number of cancelled bookings
cancelled_count = (
    df["is_canceled"] == 1
).sum()

# Total number of bookings
n_bookings = df["is_canceled"].notna().sum()

# Observed sample proportion
observed_cancellation_rate = (
    cancelled_count / n_bookings
)

# One-sample population proportion z-test
z_stat_1, p_value_1 = proportions_ztest(
    count=cancelled_count,
    nobs=n_bookings,
    value=p0,
    alternative="two-sided"
)

print_result(
    test_name="Test 1 — One-sample population proportion z-test",
    h0="p = 0.35",
    h1="p ≠ 0.35",
    statistic_name="z",
    statistic=z_stat_1,
    p_value=p_value_1
)

print(
    f"\nObserved cancellation rate = "
    f"{observed_cancellation_rate * 100:.2f}%"
)

print(
    f"Reference cancellation rate = "
    f"{p0 * 100:.2f}%"
)

print(
    f"Cancelled bookings = "
    f"{cancelled_count}"
)

print(
    f"Total bookings = "
    f"{n_bookings}"
)

Test 1 — One-sample population proportion z-test
--------------------------------------------------------------------------------
H0: p = 0.35
H1: p ≠ 0.35

z = 14.8310
p-value = 9.241e-50
alpha = 0.05

Decision: Reject H0

Observed cancellation rate = 37.07%
Reference cancellation rate = 35.00%
Cancelled bookings = 44195
Total bookings = 119205


### Decision and conclusion for Test 1

The statistical decision is translated into a conclusion specific to the hotel booking data.

In addition to determining whether the observed cancellation rate is significantly different from 35%, the direction of the difference is reported.

In [ ]:
if p_value_1 < ALPHA:

    if observed_cancellation_rate > p0:
        direction = "higher"
    else:
        direction = "lower"

    conclusion_1 = (
        "The null hypothesis is rejected. "
        f"The observed cancellation rate "
        f"({observed_cancellation_rate * 100:.2f}%) "
        f"is statistically significantly {direction} "
        f"than the reference rate of {p0 * 100:.0f}%."
    )

else:

    conclusion_1 = (
        "The null hypothesis is not rejected. "
        "There is insufficient statistical evidence "
        f"to conclude that the cancellation rate differs "
        f"from {p0 * 100:.0f}%."
    )

print("Conclusion:")
print(conclusion_1)

Conclusion:
The null hypothesis is rejected. The observed cancellation rate (37.07%) is statistically significantly higher than the reference rate of 35%.


# Test 2 — Two independent samples test

## Research question

Is the mean `lead_time` significantly different between cancelled and non-cancelled hotel bookings?

`lead_time` represents the number of days between the booking date and the scheduled arrival date.

Two independent groups are compared:

- cancelled bookings: `is_canceled = 1`,
- non-cancelled bookings: `is_canceled = 0`.

Let:

$$
\mu_{cancelled}
$$

denote the population mean lead time for cancelled bookings, and:

$$
\mu_{not\ cancelled}
$$

denote the population mean lead time for non-cancelled bookings.

The main hypotheses are:

$$
H_0: \mu_{cancelled} = \mu_{not\ cancelled}
$$

$$
H_1: \mu_{cancelled} \neq \mu_{not\ cancelled}
$$

## Why this test?

The objective is to compare the mean value of a numerical variable (`lead_time`) between two independent groups.

Therefore, an **independent-samples t-test** is appropriate.

Before the main test is performed, assumptions relevant to the choice of the t-test variant are examined using:

1. the **Shapiro-Wilk test** as a normality diagnostic,
2. **Levene's test** for equality of variances.

These are auxiliary diagnostic tests and do **not** represent additional main hypotheses required by the assignment.

### Preparation of the two independent samples

The dataset is divided into two independent samples according to cancellation status.

Descriptive statistics are calculated for each group, including:

- sample size,
- mean,
- median,
- standard deviation,
- variance.

These statistics provide an initial description of differences in `lead_time` before formal hypothesis testing.

In [ ]:
lead_time_cancelled = (
    df.loc[
        df["is_canceled"] == 1,
        "lead_time"
    ]
    .dropna()
)

lead_time_not_cancelled = (
    df.loc[
        df["is_canceled"] == 0,
        "lead_time"
    ]
    .dropna()
)


lead_time_summary = pd.DataFrame({
    "Group": [
        "Cancelled",
        "Not cancelled"
    ],
    "n": [
        len(lead_time_cancelled),
        len(lead_time_not_cancelled)
    ],
    "Mean": [
        lead_time_cancelled.mean(),
        lead_time_not_cancelled.mean()
    ],
    "Median": [
        lead_time_cancelled.median(),
        lead_time_not_cancelled.median()
    ],
    "Standard deviation": [
        lead_time_cancelled.std(ddof=1),
        lead_time_not_cancelled.std(ddof=1)
    ],
    "Variance": [
        lead_time_cancelled.var(ddof=1),
        lead_time_not_cancelled.var(ddof=1)
    ]
})

display(
    lead_time_summary.round(2)
)

,Group,n,Mean,Median,Standard deviation,Variance
0,Cancelled,44195,144.90,113.0,118.61,14068.21
1,Not cancelled,75010,80.08,45.0,91.14,8306.03


## Normality diagnostic — Shapiro-Wilk test

The example provided during classes evaluates normality before performing an independent-samples t-test. The same methodological structure is followed here.

For each cancellation group, the Shapiro-Wilk hypotheses are:

$$
H_0: \text{the population distribution is normal}
$$

$$
H_1: \text{the population distribution is not normal}
$$

The Hotel Booking Demand dataset contains tens of thousands of observations. With very large samples, formal normality tests are extremely sensitive and can detect even minor deviations from normality.

For this reason, the Shapiro-Wilk test is applied to a reproducible random sample of up to 5,000 observations from each group. It is used as a **diagnostic**, not as the main hypothesis test.

The final comparison of mean lead times still uses all available observations.

In [ ]:
SHAPIRO_SAMPLE_SIZE = 5000


cancelled_shapiro_sample = (
    lead_time_cancelled.sample(
        n=min(
            SHAPIRO_SAMPLE_SIZE,
            len(lead_time_cancelled)
        ),
        random_state=42
    )
)

not_cancelled_shapiro_sample = (
    lead_time_not_cancelled.sample(
        n=min(
            SHAPIRO_SAMPLE_SIZE,
            len(lead_time_not_cancelled)
        ),
        random_state=42
    )
)


shapiro_cancelled_stat, shapiro_cancelled_p = (
    stats.shapiro(
        cancelled_shapiro_sample
    )
)

shapiro_not_cancelled_stat, shapiro_not_cancelled_p = (
    stats.shapiro(
        not_cancelled_shapiro_sample
    )
)


print_result(
    test_name="Shapiro-Wilk test — cancelled bookings",
    h0="The population distribution is normal",
    h1="The population distribution is not normal",
    statistic_name="W",
    statistic=shapiro_cancelled_stat,
    p_value=shapiro_cancelled_p
)

print()

print_result(
    test_name="Shapiro-Wilk test — non-cancelled bookings",
    h0="The population distribution is normal",
    h1="The population distribution is not normal",
    statistic_name="W",
    statistic=shapiro_not_cancelled_stat,
    p_value=shapiro_not_cancelled_p
)

Shapiro-Wilk test — cancelled bookings
--------------------------------------------------------------------------------
H0: The population distribution is normal
H1: The population distribution is not normal

W = 0.9145
p-value = 1.301e-46
alpha = 0.05

Decision: Reject H0

Shapiro-Wilk test — non-cancelled bookings
--------------------------------------------------------------------------------
H0: The population distribution is normal
H1: The population distribution is not normal

W = 0.8326
p-value = 2.845e-58
alpha = 0.05

Decision: Reject H0


## Equality of variances — Levene's test

The class example checks equality of population variances before selecting the appropriate form of the independent-samples t-test.

Levene's test evaluates:

$$
H_0: \sigma^2_{cancelled} = \sigma^2_{not\ cancelled}
$$

$$
H_1: \sigma^2_{cancelled} \neq \sigma^2_{not\ cancelled}
$$

The result determines which form of the t-test is used:

- if $p \geq 0.05$, equal variances are not rejected and the standard independent-samples t-test can be used,
- if $p < 0.05$, equality of variances is rejected and Welch's independent-samples t-test is used.

Welch's test is appropriate when the two groups cannot be assumed to have equal population variances.

In [ ]:
levene_stat, levene_p = stats.levene(
    lead_time_cancelled,
    lead_time_not_cancelled,
    center="median"
)

print_result(
    test_name="Levene's test for equality of variances",
    h0="The population variances are equal",
    h1="The population variances are different",
    statistic_name="Levene statistic",
    statistic=levene_stat,
    p_value=levene_p
)

Levene's test for equality of variances
--------------------------------------------------------------------------------
H0: The population variances are equal
H1: The population variances are different

Levene statistic = 3579.9219
p-value = < 1e-300
alpha = 0.05

Decision: Reject H0


## Main Test 2 — Independent-samples t-test

Based on the result of Levene's test, the appropriate version of the independent-samples t-test is selected automatically.

The main hypotheses remain:

$$
H_0: \mu_{cancelled} = \mu_{not\ cancelled}
$$

$$
H_1: \mu_{cancelled} \neq \mu_{not\ cancelled}
$$

If equality of variances is rejected, Welch's independent-samples t-test is used.

The test statistic and p-value are calculated using the complete samples of cancelled and non-cancelled bookings.

In [ ]:
if levene_p < ALPHA:

    equal_variances = False

    selected_test = (
        "Welch's independent samples t-test "
        "(equal variances not assumed)"
    )

else:

    equal_variances = True

    selected_test = (
        "Independent samples t-test "
        "(equal variances assumed)"
    )


t_stat_2, p_value_2 = stats.ttest_ind(
    lead_time_cancelled,
    lead_time_not_cancelled,
    equal_var=equal_variances,
    alternative="two-sided"
)


print("Selected test:", selected_test)
print()


print_result(
    test_name="Test 2 — Two independent samples t-test",
    h0="μ_cancelled = μ_not_cancelled",
    h1="μ_cancelled ≠ μ_not_cancelled",
    statistic_name="t",
    statistic=t_stat_2,
    p_value=p_value_2
)

Selected test: Welch's independent samples t-test (equal variances not assumed)

Test 2 — Two independent samples t-test
--------------------------------------------------------------------------------
H0: μ_cancelled = μ_not_cancelled
H1: μ_cancelled ≠ μ_not_cancelled

t = 98.9564
p-value = < 1e-300
alpha = 0.05

Decision: Reject H0


### Difference between group means

The mean lead time is calculated separately for cancelled and non-cancelled bookings.

The difference between the two sample means is also calculated to support practical interpretation of the statistical result.

In [ ]:
mean_cancelled = (
    lead_time_cancelled.mean()
)

mean_not_cancelled = (
    lead_time_not_cancelled.mean()
)

mean_difference = (
    mean_cancelled
    - mean_not_cancelled
)


print(
    f"Mean lead time — cancelled bookings: "
    f"{mean_cancelled:.2f} days"
)

print(
    f"Mean lead time — non-cancelled bookings: "
    f"{mean_not_cancelled:.2f} days"
)

print(
    f"Mean difference: "
    f"{mean_difference:.2f} days"
)

Mean lead time — cancelled bookings: 144.90 days
Mean lead time — non-cancelled bookings: 80.08 days
Mean difference: 64.82 days


### Decision and conclusion for Test 2

The p-value from the independent-samples t-test is compared with alpha = 0.05.

The conclusion reports:

- whether the difference in mean lead time is statistically significant,
- which group has the higher mean lead time,
- the magnitude of the observed difference in days.

In [ ]:
if p_value_2 < ALPHA:

    if mean_cancelled > mean_not_cancelled:

        conclusion_2 = (
            "The null hypothesis is rejected. "
            "There is a statistically significant difference "
            "in mean lead time between cancelled and "
            "non-cancelled bookings. "
            f"Cancelled bookings are made, on average, "
            f"{abs(mean_difference):.2f} days further "
            "in advance than non-cancelled bookings."
        )

    else:

        conclusion_2 = (
            "The null hypothesis is rejected. "
            "There is a statistically significant difference "
            "in mean lead time between cancelled and "
            "non-cancelled bookings. "
            f"Non-cancelled bookings are made, on average, "
            f"{abs(mean_difference):.2f} days further "
            "in advance than cancelled bookings."
        )

else:

    conclusion_2 = (
        "The null hypothesis is not rejected. "
        "There is insufficient statistical evidence "
        "to conclude that mean lead time differs between "
        "cancelled and non-cancelled bookings."
    )


print("Conclusion:")
print(conclusion_2)

Conclusion:
The null hypothesis is rejected. There is a statistically significant difference in mean lead time between cancelled and non-cancelled bookings. Cancelled bookings are made, on average, 64.82 days further in advance than non-cancelled bookings.


# Test 3 — Chi-square test of independence

## Research question

Is hotel type associated with booking cancellation status?

Both variables are categorical:

- `hotel`:
  - City Hotel,
  - Resort Hotel,

- `is_canceled`:
  - `0` — not cancelled,
  - `1` — cancelled.

The hypotheses are:

$$
H_0: \text{Hotel type and cancellation status are independent}
$$

$$
H_1: \text{Hotel type and cancellation status are associated}
$$

## Why this test?

The research question concerns the relationship between **two categorical variables**.

Therefore, the **Chi-square test of independence** is appropriate.

Before performing the test, a contingency table of observed frequencies is constructed.

## Observed frequencies

A contingency table is created to show the observed number of cancelled and non-cancelled bookings for each hotel type.

This table constitutes the observed frequencies used in the Chi-square test.

### Observed frequencies

A contingency table is created to show the observed number of cancelled and non-cancelled bookings for each hotel type.

This table constitutes the observed frequencies used in the Chi-square test.

In [ ]:
chi_square_table = pd.crosstab(
    df["hotel"],
    df["is_canceled"]
)

display_table = (
    chi_square_table
    .rename(
        columns={
            0: "Not cancelled",
            1: "Cancelled"
        }
    )
)

print("Observed frequencies:")
display(display_table)

Observed frequencies:


is_canceled,Not cancelled,Cancelled
hotel,,
City Hotel,46084,33075
Resort Hotel,28926,11120


### Cancellation percentages by hotel type

Because City Hotel and Resort Hotel have different total numbers of bookings, percentages are calculated within each hotel type.

These percentages are not used directly to calculate the Chi-square statistic. They are presented to make the final result easier to interpret in the context of hotel cancellations.

In [ ]:
hotel_cancellation_percentages = (
    pd.crosstab(
        df["hotel"],
        df["is_canceled"],
        normalize="index"
    )
    * 100
)

hotel_cancellation_percentages = (
    hotel_cancellation_percentages
    .rename(
        columns={
            0: "Not cancelled [%]",
            1: "Cancelled [%]"
        }
    )
)

print(
    "Cancellation percentages within each hotel type:"
)

display(
    hotel_cancellation_percentages.round(2)
)

Cancellation percentages within each hotel type:


is_canceled,Not cancelled [%],Cancelled [%]
hotel,,
City Hotel,58.22,41.78
Resort Hotel,72.23,27.77


## Main Test 3 — Chi-square test of independence

The Chi-square test compares the observed frequencies with the frequencies that would be expected if hotel type and cancellation status were independent.

The test returns:

- the Chi-square statistic $\chi^2$,
- the p-value,
- the number of degrees of freedom,
- expected frequencies.

The p-value is compared with the significance level:

$$
\alpha = 0.05
$$

In [ ]:
chi2_stat_3, p_value_3, dof_3, expected_3 = (
    stats.chi2_contingency(
        chi_square_table
    )
)


print_result(
    test_name="Test 3 — Chi-square test of independence",
    h0="Hotel type and cancellation status are independent",
    h1="Hotel type and cancellation status are associated",
    statistic_name="χ²",
    statistic=chi2_stat_3,
    p_value=p_value_3
)

print(
    f"\nDegrees of freedom = {dof_3}"
)

Test 3 — Chi-square test of independence
--------------------------------------------------------------------------------
H0: Hotel type and cancellation status are independent
H1: Hotel type and cancellation status are associated

χ² = 2238.3436
p-value = < 1e-300
alpha = 0.05

Decision: Reject H0

Degrees of freedom = 1


### Verification of expected frequencies

The Chi-square approximation requires sufficiently large expected cell frequencies.

Expected frequencies represent the counts that would be expected in each cell if the null hypothesis of independence were true.

As a practical check, the minimum expected frequency is examined. If all expected frequencies are at least 5, the usual Chi-square approximation is considered appropriate.

In [ ]:
expected_frequencies = pd.DataFrame(
    expected_3,
    index=chi_square_table.index,
    columns=[
        "Not cancelled",
        "Cancelled"
    ]
)

print("Expected frequencies:")

display(
    expected_frequencies.round(2)
)


minimum_expected = (
    expected_frequencies
    .min()
    .min()
)

print(
    f"Minimum expected frequency = "
    f"{minimum_expected:.2f}"
)


if minimum_expected >= 5:

    print(
        "The expected-frequency condition "
        "for the Chi-square test is satisfied."
    )

else:

    print(
        "Warning: At least one expected "
        "frequency is below 5."
    )

Expected frequencies:


,Not cancelled,Cancelled
hotel,,
City Hotel,49810.97,29348.03
Resort Hotel,25199.03,14846.97


Minimum expected frequency = 14846.97
The expected-frequency condition for the Chi-square test is satisfied.


### Effect size — Cramér's V

A statistically significant Chi-square test indicates that an association exists, but it does not describe the strength of that association.

Because the Hotel Booking Demand dataset contains a very large number of observations, even relatively small associations may produce very small p-values.

Therefore, Cramér's V is calculated as an additional effect-size measure.

For a \(2 \times 2\) contingency table, values closer to 0 indicate a weaker association, while larger values indicate a stronger association.

Cramér's V is supplementary and is not required to make the reject/do-not-reject decision.

In [ ]:
n_chi = (
    chi_square_table
    .to_numpy()
    .sum()
)

rows, columns = (
    chi_square_table.shape
)

cramers_v = np.sqrt(
    chi2_stat_3
    /
    (
        n_chi
        *
        min(
            rows - 1,
            columns - 1
        )
    )
)

print(
    f"Cramér's V = {cramers_v:.4f}"
)

Cramér's V = 0.1370


### Decision and conclusion for Test 3

The statistical decision is based on the Chi-square p-value.

The final conclusion also reports the cancellation percentage for each hotel type and Cramér's V so that both statistical significance and the practical strength of the relationship can be interpreted.

In [ ]:
if p_value_3 < ALPHA:

    city_rate = (
        hotel_cancellation_percentages
        .loc[
            "City Hotel",
            "Cancelled [%]"
        ]
    )

    resort_rate = (
        hotel_cancellation_percentages
        .loc[
            "Resort Hotel",
            "Cancelled [%]"
        ]
    )

    conclusion_3 = (
        "The null hypothesis is rejected. "
        "There is a statistically significant association "
        "between hotel type and cancellation status. "
        f"The cancellation rate is {city_rate:.2f}% "
        f"for City Hotel and {resort_rate:.2f}% "
        f"for Resort Hotel. "
        f"Cramér's V is {cramers_v:.4f}."
    )

else:

    conclusion_3 = (
        "The null hypothesis is not rejected. "
        "There is insufficient statistical evidence "
        "to conclude that hotel type and "
        "cancellation status are associated."
    )


print("Conclusion:")
print(conclusion_3)

Conclusion:
The null hypothesis is rejected. There is a statistically significant association between hotel type and cancellation status. The cancellation rate is 41.78% for City Hotel and 27.77% for Resort Hotel. Cramér's V is 0.1370.


# Summary of the three main hypothesis tests

The table below summarizes only the **three main hypothesis tests required by the assignment**.

The Shapiro-Wilk and Levene tests performed earlier are diagnostic tests supporting the two-sample analysis and therefore are not listed as separate project hypotheses.

In [ ]:
results_summary = pd.DataFrame({

    "Hypothesis": [
        "Cancellation rate differs from 35%",
        "Mean lead time differs by cancellation status",
        "Hotel type is associated with cancellation status"
    ],

    "Test": [
        "One-sample population proportion z-test",
        selected_test,
        "Chi-square test of independence"
    ],

    "Statistic": [
        z_stat_1,
        t_stat_2,
        chi2_stat_3
    ],

    "p-value": [
        format_p_value(p_value_1),
        format_p_value(p_value_2),
        format_p_value(p_value_3)
    ],

    "Alpha": [
        ALPHA,
        ALPHA,
        ALPHA
    ],

    "Decision": [
        decision(p_value_1),
        decision(p_value_2),
        decision(p_value_3)
    ]
})


display(
    results_summary.round({
        "Statistic": 4
    })
)

,Hypothesis,Test,Statistic,p-value,Alpha,Decision
0,Cancellation rate differs from 35%,One-sample population proportion z-test,14.8310,9.241e-50,0.05,Reject H0
1,Mean lead time differs by cancellation status,Welch's independent samples t-test (equal vari...,98.9564,< 1e-300,0.05,Reject H0
2,Hotel type is associated with cancellation status,Chi-square test of independence,2238.3436,< 1e-300,0.05,Reject H0


# Final conclusions

The final conclusions below summarize the statistical decisions and their interpretation in the context of the Hotel Booking Demand dataset.

In [ ]:
print("FINAL CONCLUSIONS")
print("=" * 80)

print("\n1. One-sample population proportion test")
print(conclusion_1)

print("\n2. Two independent samples test")
print(conclusion_2)

print("\n3. Chi-square test of independence")
print(conclusion_3)

FINAL CONCLUSIONS

1. One-sample population proportion test
The null hypothesis is rejected. The observed cancellation rate (37.07%) is statistically significantly higher than the reference rate of 35%.

2. Two independent samples test
The null hypothesis is rejected. There is a statistically significant difference in mean lead time between cancelled and non-cancelled bookings. Cancelled bookings are made, on average, 64.82 days further in advance than non-cancelled bookings.

3. Chi-square test of independence
The null hypothesis is rejected. There is a statistically significant association between hotel type and cancellation status. The cancellation rate is 41.78% for City Hotel and 27.77% for Resort Hotel. Cramér's V is 0.1370.
